# 🔍 LLM Hallucination Detector & Auto-Corrector

## A Meta-AI System That Audits Other AI Systems

**Problem:** Large Language Models generate confident-sounding text that is sometimes factually wrong —
a phenomenon known as *hallucination*. In enterprise settings, undetected hallucinations can
erode trust, spread misinformation, and cause real-world harm.

**Solution:** This notebook builds an automated pipeline that:

1. **Decomposes** an LLM response into individual claims
2. **Retrieves** relevant evidence from a knowledge base
3. **Verifies** each claim against the evidence using NLI (Natural Language Inference)
4. **Scores** overall response reliability with a confidence rating
5. **Auto-corrects** hallucinated claims using grounded evidence

```
LLM Response → Claim Extraction → Evidence Retrieval → NLI Verification → Auto-Correction
```

**Skills Demonstrated:** NLI, Semantic Search, RAG, Prompt Engineering, Evaluation Metrics



---
## Step 1: Install Dependencies & Configuration

We install a focused set of libraries:
- `sentence-transformers` — encodes text into embeddings for semantic search
- `transformers` + `torch` — loads the NLI model for entailment checking
- `openai` / `anthropic` — LLM APIs for claim extraction and correction
- `matplotlib` — visualization of detection results

In [2]:
# ============================================================
# INSTALL DEPENDENCIES (run once)
# ============================================================

!pip install sentence-transformers transformers openai anthropic \
    python-dotenv torch matplotlib --quiet

print("\n\u2705 All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 7.7 MB/s eta 0:00:00

✅ All packages installed!


In [3]:
# ============================================================
# CONFIGURATION — Single source of truth
# ============================================================
# Change provider, models, and thresholds here.
# The rest of the notebook adapts automatically.
# ============================================================

import os
from dotenv import load_dotenv

load_dotenv()  # Load .env file if present

# --- API Keys ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-key-here")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-anthropic-key-here")

# --- LLM Provider for claim extraction & correction ---
LLM_PROVIDER = "openai"                           # "openai" or "anthropic"
OPENAI_MODEL = "gpt-4o"                           # OpenAI model
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"      # Anthropic model

# --- Embedding Model for semantic search ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"              # Fast & lightweight

# --- NLI Model for entailment verification ---
# Classifies (evidence, claim) as ENTAILMENT / CONTRADICTION / NEUTRAL
NLI_MODEL = "cross-encoder/nli-deberta-v3-small"

# --- Verification Thresholds ---
RETRIEVAL_TOP_K = 3              # Evidence chunks to retrieve per claim
ENTAILMENT_THRESHOLD = 0.6       # Above this = SUPPORTED
CONTRADICTION_THRESHOLD = 0.5    # Above this = CONTRADICTED

print("\u2705 Configuration loaded!")
print(f"   LLM Provider:    {LLM_PROVIDER}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   NLI Model:       {NLI_MODEL}")

✅ Configuration loaded!
   LLM Provider:    openai
   Embedding Model: all-MiniLM-L6-v2
   NLI Model:       cross-encoder/nli-deberta-v3-small


---
## Step 2: Initialize Models & API Clients

We load three core components:

1. **Embedding Model** (`all-MiniLM-L6-v2`) — Converts text into vectors for cosine similarity retrieval. Only 80MB, fast, and accurate.

2. **NLI Cross-Encoder** (`nli-deberta-v3-small`) — Takes a (premise, hypothesis) pair and outputs probabilities for entailment, contradiction, and neutral. This is the *core verification engine*.

3. **LLM Client** — Used for claim extraction and auto-correction (tasks that require reasoning).

In [4]:
# ============================================================
# INITIALIZE MODELS & CLIENTS
# ============================================================

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from openai import OpenAI
import anthropic

# 1. Embedding Model
print("\u23f3 Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"   \u2713 Loaded: {EMBEDDING_MODEL}")

# 2. NLI Cross-Encoder
# Given (evidence, claim), outputs P(entailment), P(contradiction), P(neutral)
print("\u23f3 Loading NLI model...")
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
nli_model.eval()  # Evaluation mode (disables dropout)
print(f"   \u2713 Loaded: {NLI_MODEL}")

# 3. LLM Clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"   \u2713 LLM client ready ({LLM_PROVIDER})")

print("\n\u2705 All models initialized!")

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✓ Loaded: cross-encoder/nli-deberta-v3-small
   ✓ LLM client ready (openai)

✅ All models initialized!


---
## Step 3: Create Knowledge Base & Vector Index

The knowledge base is our **source of truth**. Every claim from the LLM will be verified against these documents.

**Indexing pipeline:**
1. Define documents inline (or load from JSON if the `data/` folder exists)
2. Split each document into sentence-level chunks
3. Encode all chunks into embeddings
4. Build an in-memory cosine-similarity vector index

In production, you'd replace this with ChromaDB, Pinecone, or Weaviate.

In [5]:
# ============================================================
# KNOWLEDGE BASE & VECTOR INDEX
# ============================================================
# The knowledge base is embedded inline so this notebook is
# fully self-contained — no external files needed.
# ============================================================

import json
import re
from dataclasses import dataclass, field
from typing import List, Tuple

# --- Knowledge Base Documents (inline) ---
# 10 documents covering AI/ML/Software topics with verified facts.
# Each sentence is a potential evidence chunk for verification.

KB_DOCUMENTS = [
    {
        "id": "doc_001", "title": "Python Programming Language",
        "content": "Python is a high-level, general-purpose programming language created by Guido van Rossum. It was first released in 1991. Python's design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically typed and garbage-collected. It supports multiple programming paradigms, including structured, object-oriented and functional programming. The Python Software Foundation manages and directs resources for Python development. Python 3.0, released in 2008, was a major revision not completely backward-compatible with earlier versions. Python 2 was officially discontinued in 2020."
    },
    {
        "id": "doc_002", "title": "Transformer Architecture in AI",
        "content": "The Transformer is a deep learning architecture introduced in the 2017 paper 'Attention Is All You Need' by Vaswani et al. at Google. Unlike recurrent neural networks (RNNs), Transformers process all input data simultaneously using self-attention mechanisms. The original Transformer was designed for machine translation tasks. Key components include multi-head self-attention, positional encoding, and feed-forward neural networks. BERT, released by Google in 2018, uses only the encoder part of the Transformer. GPT models from OpenAI use only the decoder part. The Transformer architecture has become the foundation for most modern large language models including GPT-4, Claude, Gemini, and LLaMA."
    },
    {
        "id": "doc_003", "title": "Machine Learning Fundamentals",
        "content": "Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, models are trained on labeled data with input-output pairs. Common supervised learning algorithms include linear regression, decision trees, random forests, and support vector machines. Unsupervised learning works with unlabeled data to find hidden patterns. K-means clustering and principal component analysis (PCA) are popular unsupervised methods. Reinforcement learning involves agents learning through trial and error by interacting with an environment. Deep learning is a subset of machine learning that uses neural networks with multiple layers."
    },
    {
        "id": "doc_004", "title": "Natural Language Processing",
        "content": "Natural Language Processing (NLP) is a field at the intersection of computer science, artificial intelligence, and linguistics. NLP focuses on enabling computers to understand, interpret, and generate human language. Key NLP tasks include tokenization, part-of-speech tagging, named entity recognition, sentiment analysis, machine translation, and text summarization. Word embeddings like Word2Vec, introduced by Google in 2013, represent words as dense vectors. GloVe, developed at Stanford, is another widely used word embedding method. The introduction of Transformers in 2017 revolutionized NLP. Pre-trained language models like BERT and GPT have achieved state-of-the-art results on most NLP benchmarks."
    },
    {
        "id": "doc_005", "title": "Cloud Computing and AWS",
        "content": "Amazon Web Services (AWS) is the world's most widely adopted cloud platform, launched in 2006. AWS offers over 200 fully featured services from data centers globally. Key services include EC2 for compute, S3 for storage, RDS for databases, and Lambda for serverless computing. AWS operates in multiple geographic regions, each containing multiple availability zones. Other major cloud providers include Microsoft Azure, launched in 2010, and Google Cloud Platform (GCP), launched in 2011. Cloud computing follows three main service models: Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). AWS holds approximately 31% of the global cloud infrastructure market share as of 2024."
    },
    {
        "id": "doc_006", "title": "Neural Network Architectures",
        "content": "A neural network is a computational model inspired by the structure of biological neural networks. The perceptron, invented by Frank Rosenblatt in 1958, is the simplest form of a neural network. Convolutional Neural Networks (CNNs), pioneered by Yann LeCun in 1989, are primarily used for image recognition tasks. Recurrent Neural Networks (RNNs) are designed for sequential data processing. Long Short-Term Memory (LSTM) networks, introduced by Hochreiter and Schmidhuber in 1997, solved the vanishing gradient problem in RNNs. Generative Adversarial Networks (GANs) were introduced by Ian Goodfellow in 2014. Diffusion models, which power systems like Stable Diffusion and DALL-E, became prominent in 2022 for image generation."
    },
    {
        "id": "doc_007", "title": "Data Science Tools and Libraries",
        "content": "Pandas is an open-source data analysis library for Python, created by Wes McKinney in 2008. NumPy provides support for large multi-dimensional arrays and matrices in Python. Scikit-learn is a machine learning library that provides simple and efficient tools for data mining and analysis. TensorFlow, developed by Google Brain, was released as open source in 2015. PyTorch, developed by Meta AI (formerly Facebook AI Research), was released in 2016 and has become the most popular deep learning framework in research. Matplotlib is a plotting library for Python. Jupyter Notebook is an open-source web application for creating documents with live code. Apache Spark is a unified analytics engine for large-scale data processing."
    },
    {
        "id": "doc_008", "title": "Large Language Models",
        "content": "Large Language Models (LLMs) are neural networks trained on vast amounts of text data. GPT-3, released by OpenAI in June 2020, had 175 billion parameters. GPT-4, released in March 2023, is a multimodal model accepting both text and image inputs. Claude is developed by Anthropic, a company founded in 2021 by former OpenAI researchers including Dario and Daniela Amodei. Google's Gemini (formerly Bard) is powered by the Gemini model family. LLaMA (Large Language Model Meta AI) was released by Meta. LLMs are trained using techniques like next-token prediction and reinforcement learning from human feedback (RLHF). Common challenges include hallucination, bias, and high computational costs. Context window sizes have grown from 2048 tokens in GPT-2 to over 100,000 tokens in modern models."
    },
    {
        "id": "doc_009", "title": "Version Control with Git",
        "content": "Git is a distributed version control system created by Linus Torvalds in 2005 for Linux kernel development. GitHub, launched in 2008, is the largest host for Git repositories and was acquired by Microsoft in 2018 for 7.5 billion dollars. GitLab is another popular Git hosting service that offers integrated CI/CD pipelines. Key Git concepts include repositories, branches, commits, merges, and pull requests. The default branch in new Git repositories was traditionally called 'master' but many projects have switched to 'main'. Git uses SHA-1 hashing to identify objects. Common Git workflows include GitFlow, GitHub Flow, and trunk-based development."
    },
    {
        "id": "doc_010", "title": "Retrieval-Augmented Generation",
        "content": "Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant information from external knowledge bases before generating answers. RAG was introduced in a 2020 paper by Lewis et al. at Facebook AI Research. The typical RAG pipeline involves document chunking, embedding generation, vector storage, retrieval, and augmented generation. Popular vector databases include Pinecone, Weaviate, ChromaDB, and Milvus. Embedding models convert text into dense vector representations for similarity search. Common chunking strategies include fixed-size chunking, semantic chunking, and recursive character splitting. RAG helps reduce hallucinations by grounding LLM responses in factual source documents. GraphRAG extends traditional RAG by incorporating knowledge graph structures for better contextual understanding."
    }
]

print(f"\u2705 Knowledge base loaded: {len(KB_DOCUMENTS)} documents")

✅ Knowledge base loaded: 10 documents


In [10]:
# ============================================================
# VECTOR INDEX CLASS
# ============================================================
# Splits documents into sentences, embeds them, and provides
# cosine-similarity search. This is a minimal RAG retriever.
# ============================================================

@dataclass
class KnowledgeChunk:
    """A single retrievable unit of knowledge."""
    text: str
    source_doc_id: str
    source_title: str
    chunk_index: int


class VectorIndex:
    """
    Minimal in-memory vector search index.
    In production, replace with ChromaDB / Pinecone / Weaviate.
    """

    def __init__(self, emb_model):
        self.emb_model = emb_model
        self.chunks: List[KnowledgeChunk] = []
        self.embeddings = None  # np.ndarray, shape (n, dim)

    def add_documents(self, documents: list):
        """Split docs into sentences, embed, and index."""
        for doc in documents:
            sentences = re.split(r'(?<=[.!?])\s+', doc["content"])
            for i, sent in enumerate(sentences):
                sent = sent.strip()
                if len(sent) < 10:
                    continue
                self.chunks.append(KnowledgeChunk(
                    text=sent,
                    source_doc_id=doc["id"],
                    source_title=doc["title"],
                    chunk_index=i
                ))

        # Batch-encode all chunks
        texts = [c.text for c in self.chunks]
        self.embeddings = self.emb_model.encode(
            texts, show_progress_bar=True, convert_to_numpy=True
        )
        # Normalize for cosine similarity via dot product
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.embeddings = self.embeddings / norms
        print(f"   Indexed {len(self.chunks)} chunks from {len(documents)} docs")

    def search(self, query: str, top_k: int = 3) -> List[Tuple[KnowledgeChunk, float]]:
        """Return top_k most similar chunks to the query."""
        q_emb = self.emb_model.encode([query], convert_to_numpy=True)
        q_emb = q_emb / np.linalg.norm(q_emb)
        sims = np.dot(self.embeddings, q_emb.T).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in top_idx]


# --- Build the index ---
print("\u23f3 Building vector index...")
vector_index = VectorIndex(embedding_model)
vector_index.add_documents(KB_DOCUMENTS)
print(f"\n\u2705 Knowledge base ready! ({len(vector_index.chunks)} searchable chunks)")

# --- Quick test ---
print("\n\U0001f50e Test: 'Who created Python?'")
for i, (chunk, score) in enumerate(vector_index.search("Who created Python?", 3), 1):
    print(f"   {i}. [{score:.3f}] {chunk.text}")

⏳ Building vector index...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

   Indexed 78 chunks from 10 docs

✅ Knowledge base ready! (78 searchable chunks)

🔎 Test: 'Who created Python?'
   1. [0.748] Python is a high-level, general-purpose programming language created by Guido van Rossum.
   2. [0.572] Python 2 was officially discontinued in 2020.
   3. [0.550] Python 3.0, released in 2008, was a major revision not completely backward-compatible with earlier versions.


---
## Step 4: Claim Extractor (LLM-Powered)

Before we can verify an LLM response, we break it into **individual, atomic claims** — each stating a single verifiable fact.

**Why not sentence-by-sentence?** Because one sentence can contain multiple claims:

> *"Python was created by Guido van Rossum and released in 1995."*
> - Claim 1: Python was created by Guido van Rossum ✔️
> - Claim 2: Python was released in 1995 ❌ (correct: 1991)

In [1]:
# ============================================================
# CLAIM EXTRACTOR
# ============================================================
# Uses an LLM to decompose a response into individual,
# verifiable atomic claims.
# ============================================================

import time

CLAIM_SYSTEM = """You are a precise fact-checking assistant.
Decompose text into individual, atomic, verifiable claims.

Rules:
1. Each claim must state exactly ONE verifiable fact
2. Claims must be self-contained
3. Preserve original meaning - do not add or infer information
4. Skip opinions and vague filler phrases
5. Include specific details: names, dates, numbers, relationships
6. Output ONLY numbered claims, one per line

"""
def call_llm(prompt: str, system_prompt: str, temperature: float = 0.0) -> str:
    """Provider-agnostic LLM call. Temperature 0 for deterministic output."""
    if LLM_PROVIDER == "openai":
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=1024
        )
        return response.choices[0].message.content
    elif LLM_PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=1024,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text


def extract_claims(llm_response: str) -> List[str]:
    """Extract individual verifiable claims from an LLM response."""
    prompt = f'Decompose this text into atomic verifiable claims:\n\n"""{llm_response}"""\n\nNumbered claims:'
    raw = call_llm(prompt, CLAIM_SYSTEM, temperature=0.0)

    claims = []
    for line in raw.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        cleaned = re.sub(r'^\d+[.)\-]\s*', '', line).strip()
        if len(cleaned) > 5:
            claims.append(cleaned)
    return claims


# --- Quick Test ---
test_text = "Python is a programming language created by Guido van Rossum. It was first released in 1995."
print("\U0001f4cb Extracting claims...\n")
test_claims = extract_claims(test_text)
for i, c in enumerate(test_claims, 1):
    print(f"   {i}. {c}")
print(f"\n\u2705 Extracted {len(test_claims)} claims")

---
## Step 5: NLI Verifier — The Core Verification Engine

This is the **most important part** of the project. For each claim:

1. **Retrieve** top-K evidence chunks via semantic search
2. **Run NLI** on each (evidence, claim) pair
3. **Aggregate** scores to reach a verdict

**Why NLI instead of just cosine similarity?**
Similarity only tells you if topics are *related*. NLI tells you if facts *conflict*.
Two sentences about Python's release date will have high similarity — but one might say 1991 and the other 1995. Only NLI catches the contradiction.

In [ ]:
# ============================================================
# NLI VERIFIER
# ============================================================
# Core verification pipeline:
#   Retrieve evidence -> Run NLI -> Produce verdict
# ============================================================

from enum import Enum


class Verdict(Enum):
    SUPPORTED = "\u2705 SUPPORTED"
    CONTRADICTED = "\u274c CONTRADICTED"
    UNVERIFIABLE = "\u26a0\ufe0f UNVERIFIABLE"


@dataclass
class VerificationResult:
    """Complete verification output for a single claim."""
    claim: str
    verdict: Verdict
    confidence: float
    entailment_score: float
    contradiction_score: float
    evidence: List[Tuple[str, float]]
    best_evidence: str


def run_nli(premise: str, hypothesis: str) -> dict:
    """
    Run NLI on a (premise, hypothesis) pair.
    Returns probabilities for entailment, contradiction, neutral.
    """
    inputs = nli_tokenizer(
        premise, hypothesis,
        return_tensors="pt", truncation=True, max_length=512
    )
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=1).squeeze().tolist()
    # DeBERTa label order: 0=contradiction, 1=neutral, 2=entailment
    return {"contradiction": probs[0], "neutral": probs[1], "entailment": probs[2]}


def verify_claim(claim: str, index: VectorIndex) -> VerificationResult:
    """
    Verify a single claim against the knowledge base.
    Retrieves evidence, runs NLI, and produces a verdict.
    """
    # 1. Retrieve evidence
    search_results = index.search(claim, top_k=RETRIEVAL_TOP_K)

    # 2. Run NLI on each (evidence, claim) pair
    max_ent, max_con = 0.0, 0.0
    best_evidence_text = ""
    evidence_list = []

    for chunk, sim in search_results:
        nli = run_nli(premise=chunk.text, hypothesis=claim)
        evidence_list.append((chunk.text, sim))

        if nli["entailment"] > max_ent:
            max_ent = nli["entailment"]
            if nli["entailment"] > nli["contradiction"]:
                best_evidence_text = chunk.text

        if nli["contradiction"] > max_con:
            max_con = nli["contradiction"]
            if nli["contradiction"] > nli["entailment"]:
                best_evidence_text = chunk.text

    # 3. Determine verdict
    if max_ent >= ENTAILMENT_THRESHOLD and max_ent > max_con:
        verdict, confidence = Verdict.SUPPORTED, max_ent
    elif max_con >= CONTRADICTION_THRESHOLD and max_con > max_ent:
        verdict, confidence = Verdict.CONTRADICTED, max_con
    else:
        verdict, confidence = Verdict.UNVERIFIABLE, 1.0 - max(max_ent, max_con)

    if not best_evidence_text and evidence_list:
        best_evidence_text = evidence_list[0][0]

    return VerificationResult(
        claim=claim, verdict=verdict, confidence=confidence,
        entailment_score=max_ent, contradiction_score=max_con,
        evidence=evidence_list, best_evidence=best_evidence_text
    )


# --- Quick Test ---
print("\U0001f50e Testing on a CORRECT claim...")
r1 = verify_claim("Python was created by Guido van Rossum.", vector_index)
print(f"   {r1.verdict.value} (conf: {r1.confidence:.3f})")

print("\n\U0001f50e Testing on a HALLUCINATED claim...")
r2 = verify_claim("Python was first released in 1995.", vector_index)
print(f"   {r2.verdict.value} (conf: {r2.confidence:.3f})")
print(f"   Evidence: {r2.best_evidence}")

---
## Step 6: Auto-Corrector

When a claim is flagged as **CONTRADICTED**, we don't just stop at detection — we **auto-correct** it using the evidence that contradicted it. The corrector preserves the original sentence structure and changes only the incorrect parts.

In [ ]:
# ============================================================
# AUTO-CORRECTOR
# ============================================================
# Rewrites hallucinated claims using contradicting evidence.
# ============================================================

CORRECTION_SYSTEM = """You are a fact-correction assistant.
Given an incorrect claim and correct evidence, rewrite the claim accurately.

Rules:
1. Preserve the original sentence structure
2. Change ONLY the incorrect parts
3. Base corrections SOLELY on the evidence
4. Output ONLY the corrected claim"""


def auto_correct(claim: str, evidence: str) -> str:
    """Generate a corrected version of a hallucinated claim."""
    prompt = f'INCORRECT: "{claim}"\nEVIDENCE: "{evidence}"\n\nCorrected claim:'
    return call_llm(prompt, CORRECTION_SYSTEM, temperature=0.0).strip().strip('"')


# --- Quick Test ---
print("\U0001f527 Testing auto-correction...\n")
corrected = auto_correct(
    "Python was first released in 1995.",
    "It was first released in 1991."
)
print(f"   Original:  Python was first released in 1995.")
print(f"   Corrected: {corrected}")

---
## Step 7: Full Pipeline — The HallucinationDetector Class

This assembles everything into a single `.audit()` method:

```
LLM Response → Extract Claims → Verify Each → Auto-Correct → Audit Report
```

In [ ]:
# ============================================================
# FULL HALLUCINATION DETECTION PIPELINE
# ============================================================

@dataclass
class AuditReport:
    """Complete hallucination audit report."""
    original_text: str
    claims: List[str]
    results: List[VerificationResult]
    corrections: dict
    reliability_score: float
    summary: dict


class HallucinationDetector:
    """Main pipeline: audit an LLM response for hallucinations."""

    def __init__(self, index: VectorIndex):
        self.index = index

    def audit(self, llm_response: str, do_correct: bool = True) -> AuditReport:
        print("\n" + "=" * 65)
        print("\U0001f50d HALLUCINATION AUDIT")
        print("=" * 65)

        # Step 1: Extract claims
        print("\n\U0001f4cb Step 1/3: Extracting claims...")
        claims = extract_claims(llm_response)
        print(f"   Found {len(claims)} verifiable claims")

        # Step 2: Verify each claim
        print("\n\U0001f50e Step 2/3: Verifying against knowledge base...")
        results = []
        for i, claim in enumerate(claims, 1):
            result = verify_claim(claim, self.index)
            results.append(result)
            icon = result.verdict.value.split()[0]
            display_claim = claim[:65] + "..." if len(claim) > 65 else claim
            print(f"   [{i}/{len(claims)}] {icon} {display_claim}")

        # Step 3: Auto-correct hallucinations
        corrections = {}
        contradicted = [r for r in results if r.verdict == Verdict.CONTRADICTED]
        if do_correct and contradicted:
            print(f"\n\U0001f527 Step 3/3: Correcting {len(contradicted)} hallucination(s)...")
            for r in contradicted:
                corrections[r.claim] = auto_correct(r.claim, r.best_evidence)
        else:
            print("\n\U0001f527 Step 3/3: No corrections needed.")

        # Calculate reliability score
        if results:
            score_sum = sum(
                1.0 if r.verdict == Verdict.SUPPORTED else
                0.5 if r.verdict == Verdict.UNVERIFIABLE else 0.0
                for r in results
            )
            reliability = score_sum / len(results)
        else:
            reliability = 0.0

        summary = {
            "total_claims": len(claims),
            "supported": sum(1 for r in results if r.verdict == Verdict.SUPPORTED),
            "contradicted": sum(1 for r in results if r.verdict == Verdict.CONTRADICTED),
            "unverifiable": sum(1 for r in results if r.verdict == Verdict.UNVERIFIABLE),
        }

        return AuditReport(
            original_text=llm_response, claims=claims, results=results,
            corrections=corrections, reliability_score=reliability, summary=summary
        )


detector = HallucinationDetector(vector_index)
print("\n\u2705 Hallucination Detector ready!")

---
## Step 8: Report Renderer

A color-coded audit report with reliability gauge, per-claim verdicts, evidence citations, and corrections.

In [ ]:
# ============================================================
# REPORT RENDERER
# ============================================================

def render_report(report: AuditReport):
    """Display a formatted audit report."""
    s = report.summary
    score = report.reliability_score

    # Reliability gauge
    filled = int(score * 30)
    bar = "\u2588" * filled + "\u2591" * (30 - filled)
    if score >= 0.8:
        grade = "HIGH RELIABILITY"
    elif score >= 0.5:
        grade = "MODERATE RELIABILITY"
    else:
        grade = "LOW RELIABILITY \u2014 SIGNIFICANT HALLUCINATIONS"

    print("\n" + "=" * 65)
    print("\U0001f4ca AUDIT REPORT")
    print("=" * 65)
    print(f"\n   Reliability: [{bar}] {score:.0%}")
    print(f"   Grade:       {grade}")
    print(f"\n   Total Claims: {s['total_claims']}")
    print(f"   \u2705 Supported:    {s['supported']}")
    print(f"   \u274c Contradicted: {s['contradicted']}")
    print(f"   \u26a0\ufe0f  Unverifiable: {s['unverifiable']}")

    # Per-claim details
    print("\n" + "-" * 65)
    print("CLAIM-BY-CLAIM ANALYSIS")
    print("-" * 65)

    for i, r in enumerate(report.results, 1):
        print(f"\n   Claim {i}: {r.claim}")
        print(f"   Verdict:  {r.verdict.value} (confidence: {r.confidence:.2f})")
        print(f"   Scores:   entailment={r.entailment_score:.3f}  contradiction={r.contradiction_score:.3f}")
        ev_display = r.best_evidence[:85] + "..." if len(r.best_evidence) > 85 else r.best_evidence
        print(f"   Evidence: \"{ev_display}\"")
        if r.claim in report.corrections:
            print(f"   \U0001f527 Correction: {report.corrections[r.claim]}")

    # Corrections summary
    if report.corrections:
        print("\n" + "-" * 65)
        print("\U0001f4dd CORRECTIONS SUMMARY")
        print("-" * 65)
        for orig, fixed in report.corrections.items():
            print(f"   \u274c {orig}")
            print(f"   \u2705 {fixed}\n")

    print("=" * 65)


print("\u2705 Report renderer ready!")

---
## Step 9: Run the Pipeline — Live Demos

We test the detector on LLM responses with **known hallucinations** mixed with correct facts — exactly the kind of output real LLMs produce.

In [ ]:
# ============================================================
# DEMO 1: Python (3 hallucinations planted)
# ============================================================
# Errors: release year (1995 vs 1991), typing (static vs dynamic),
#         backward compatibility (fully vs not completely)
# ============================================================

demo_1 = (
    "Python is a high-level programming language created by Guido van Rossum. "
    "It was first released in 1995. Python is statically typed and uses "
    "significant indentation for code readability. Python 3.0 was released "
    "in 2008 and was fully backward-compatible with Python 2. The Python "
    "Software Foundation manages Python development. Python 2 was officially "
    "discontinued in 2020."
)

report_1 = detector.audit(demo_1)
render_report(report_1)

In [ ]:
# ============================================================
# DEMO 2: Transformers (3 hallucinations planted)
# ============================================================
# Errors: OpenAI vs Google, sequential vs simultaneous,
#         BERT full vs encoder-only
# ============================================================

demo_2 = (
    "The Transformer architecture was introduced in the 2017 paper "
    "'Attention Is All You Need' by researchers at OpenAI. It uses "
    "self-attention mechanisms to process input data sequentially, "
    "one token at a time. BERT, released by Google in 2018, uses the "
    "full encoder-decoder Transformer. GPT models from OpenAI use only "
    "the decoder part of the Transformer."
)

report_2 = detector.audit(demo_2)
render_report(report_2)

In [ ]:
# ============================================================
# DEMO 3: RAG — Fully Correct (0 hallucinations)
# ============================================================
# This response is entirely accurate. The detector should give
# it a HIGH reliability score.
# ============================================================

demo_3 = (
    "Retrieval-Augmented Generation (RAG) is a technique that enhances "
    "LLM responses by retrieving relevant information from external "
    "knowledge bases. RAG was introduced in a 2020 paper by Lewis et al. "
    "at Facebook AI Research. Popular vector databases include Pinecone, "
    "ChromaDB, and Milvus. RAG helps reduce hallucinations by grounding "
    "responses in source documents."
)

report_3 = detector.audit(demo_3)
render_report(report_3)

In [ ]:
# ============================================================
# DEMO 4: Git & GitHub (3 hallucinations planted)
# ============================================================
# Errors: GitHub 2010 vs 2008, Google vs Microsoft, MD5 vs SHA-1
# ============================================================

demo_4 = (
    "Git is a distributed version control system created by Linus Torvalds "
    "in 2005. GitHub was launched in 2010 and was acquired by Google in "
    "2020 for 10 billion dollars. GitLab offers integrated CI/CD pipelines. "
    "Git uses MD5 hashing to identify objects. Common workflows include "
    "GitFlow and GitHub Flow."
)

report_4 = detector.audit(demo_4)
render_report(report_4)

---
## Step 10: Batch Evaluation with Test Suite

We run the detector on **8 test cases** with known hallucinations and measure detection performance.

In [ ]:
# ============================================================
# BATCH EVALUATION
# ============================================================
# Test cases with known hallucinations for measuring precision.
# ============================================================

TEST_CASES = [
    {
        "id": "test_001", "question": "Python language",
        "llm_response": "Python is a high-level programming language created by Guido van Rossum. It was first released in 1995. Python is statically typed. Python 2 was discontinued in 2020.",
        "expected_hallucinations": 2,  # year, typing
        "has_errors": True
    },
    {
        "id": "test_002", "question": "Transformer architecture",
        "llm_response": "The Transformer was introduced in 2017 by researchers at OpenAI. It processes data sequentially. BERT uses the full encoder-decoder Transformer.",
        "expected_hallucinations": 3,
        "has_errors": True
    },
    {
        "id": "test_003", "question": "Machine learning types",
        "llm_response": "The four main types of ML are supervised, unsupervised, reinforcement, and transfer learning. K-means is a supervised learning method.",
        "expected_hallucinations": 2,
        "has_errors": True
    },
    {
        "id": "test_004", "question": "Git and GitHub",
        "llm_response": "Git was created by Linus Torvalds in 2005. GitHub was launched in 2010 and acquired by Google for 10 billion dollars. Git uses MD5 hashing.",
        "expected_hallucinations": 3,
        "has_errors": True
    },
    {
        "id": "test_005", "question": "RAG (fully correct)",
        "llm_response": "RAG was introduced in 2020 by Lewis et al. at Facebook AI Research. It enhances LLM responses by retrieving information from knowledge bases. Popular vector databases include Pinecone and ChromaDB.",
        "expected_hallucinations": 0,
        "has_errors": False
    },
    {
        "id": "test_006", "question": "LLMs",
        "llm_response": "GPT-4 was released in March 2023 as a text-only model. Anthropic was founded by former Google DeepMind researchers.",
        "expected_hallucinations": 2,
        "has_errors": True
    },
    {
        "id": "test_007", "question": "Cloud computing",
        "llm_response": "AWS was launched in 2006. Microsoft Azure was launched in 2008. AWS holds approximately 45% market share.",
        "expected_hallucinations": 2,
        "has_errors": True
    },
    {
        "id": "test_008", "question": "Neural networks",
        "llm_response": "The perceptron was invented by Rosenblatt in 1958. CNNs were pioneered by Yann LeCun. GANs were introduced by Yann LeCun in 2016.",
        "expected_hallucinations": 1,
        "has_errors": True
    },
]

print("\U0001f4ca BATCH EVALUATION")
print("=" * 65)
print(f"Running {len(TEST_CASES)} test cases...\n")

eval_results = []

for tc in TEST_CASES:
    print(f"--- {tc['id']}: {tc['question']} ---")
    report = detector.audit(tc["llm_response"], do_correct=False)

    detected = sum(1 for r in report.results if r.verdict == Verdict.CONTRADICTED)
    eval_results.append({
        "test_id": tc["id"],
        "has_errors": tc["has_errors"],
        "expected": tc["expected_hallucinations"],
        "detected": detected,
        "total_claims": len(report.claims),
        "reliability": report.reliability_score
    })
    print(f"   Expected: {tc['expected_hallucinations']} | Detected: {detected} | Reliability: {report.reliability_score:.2f}\n")

# Summary
print("=" * 65)
print("\U0001f4ca SUMMARY")
print("=" * 65)
total_exp = sum(r["expected"] for r in eval_results)
total_det = sum(r["detected"] for r in eval_results)
print(f"   Total expected hallucinations: {total_exp}")
print(f"   Total detected hallucinations: {total_det}")

clean = [r for r in eval_results if not r["has_errors"]]
dirty = [r for r in eval_results if r["has_errors"]]
if clean:
    print(f"   Clean responses avg reliability:  {np.mean([r['reliability'] for r in clean]):.2f}")
if dirty:
    print(f"   Error responses avg reliability:   {np.mean([r['reliability'] for r in dirty]):.2f}")

---
## Step 11: Visualization — Detection Performance Dashboard

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (16, 5)
matplotlib.rcParams['font.size'] = 11

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Reliability per test case
ax1 = axes[0]
ids = [r["test_id"].replace("test_", "#") for r in eval_results]
scores = [r["reliability"] for r in eval_results]
colors = ["#2ecc71" if not r["has_errors"] else "#e74c3c" for r in eval_results]
ax1.bar(ids, scores, color=colors, edgecolor="white")
ax1.axhline(y=0.5, color="orange", linestyle="--", alpha=0.7, label="Threshold")
ax1.set_ylabel("Reliability Score")
ax1.set_title("Reliability per Test Case")
ax1.set_ylim(0, 1.1)
ax1.legend()

# Chart 2: Expected vs Detected
ax2 = axes[1]
x = range(len(eval_results))
w = 0.35
ax2.bar([i - w/2 for i in x], [r["expected"] for r in eval_results], w,
        label="Expected", color="#3498db", edgecolor="white")
ax2.bar([i + w/2 for i in x], [r["detected"] for r in eval_results], w,
        label="Detected", color="#e67e22", edgecolor="white")
ax2.set_ylabel("Count")
ax2.set_title("Expected vs Detected Hallucinations")
ax2.set_xticks(list(x))
ax2.set_xticklabels(ids)
ax2.legend()

# Chart 3: Verdict distribution
ax3 = axes[2]
total_ok = sum(r["total_claims"] - r["detected"] for r in eval_results)
total_bad = sum(r["detected"] for r in eval_results)
ax3.pie([total_ok, total_bad],
        labels=["Supported/Unverifiable", "Contradicted"],
        colors=["#2ecc71", "#e74c3c"],
        autopct="%1.0f%%", startangle=90)
ax3.set_title("Overall Verdict Distribution")

plt.tight_layout()
plt.savefig("hallucination_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Chart saved to hallucination_results.png")

---
## Step 12: Interactive Mode — Try Your Own Text

Paste any LLM response below and run the cell to get a full hallucination audit!

In [ ]:
# ============================================================
# INTERACTIVE MODE — Audit any LLM response
# ============================================================
# Replace the text below with any LLM output you want to check.
# ============================================================

your_text = (
    "GPT-3 was released by OpenAI in 2020 with 175 billion parameters. "
    "GPT-4 was released in March 2023 as a text-only model. Claude is "
    "developed by Anthropic, founded by former Google DeepMind researchers. "
    "LLMs are trained using next-token prediction and RLHF."
)

your_report = detector.audit(your_text, do_correct=True)
render_report(your_report)

---
## Architecture Overview

```
┌──────────────────────────────────────────────────────────────┐
│       LLM HALLUCINATION DETECTOR & AUTO-CORRECTOR        │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│   LLM Response → [Claim Extractor] → Atomic Claims           │
│        │                                                      │
│        ▼                                                      │
│   [Evidence Retriever] ↔ [Vector Index / Knowledge Base]     │
│        │                                                      │
│        ▼                                                      │
│   [NLI Verifier] → SUPPORTED / CONTRADICTED / UNVERIFIABLE   │
│        │                                                      │
│        ▼  (if contradicted)                                   │
│   [Auto-Corrector] → Fixed claim grounded in evidence        │
│        │                                                      │
│        ▼                                                      │
│   [Audit Report] → Reliability score + corrections           │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

## Key Technical Decisions

1. **NLI over similarity**: Cosine similarity only tells you topics are related. NLI detects when facts *conflict*.
2. **Atomic claims**: One sentence can contain multiple facts. Decomposing enables per-fact verification.
3. **Dual-model**: Embedding model for fast retrieval + NLI model for accurate verification.
4. **Configurable thresholds**: Tune the strictness tradeoff between false positives and false negatives.
